In [13]:
"""
price_fetcher.py
----------------
Replaces yfinance with nselib for fetching daily price history — for both
individual stocks and the Nifty Total Market benchmark index.

Why: yfinance depends on curl_cffi / native extensions that have caused
segmentation faults when combined with Streamlit'''s file-watcher and thread
pool on some macOS setups (Python 3.11+/ARM). nselib is pure-Python
(requests + pandas), which avoids that entire class of crash.
"""

import logging
import time
import pandas as pd
from nselib import capital_market, indices

logger = logging.getLogger("price_fetcher")


def _to_ddmmyyyy(iso_date: str) -> str:
    return pd.Timestamp(iso_date).strftime("%d-%m-%Y")




def fetch_stock_prices(symbols: list[str], start: str, end: str) -> pd.DataFrame:
    """
    symbols: plain NSE symbols WITHOUT .NS suffix (e.g. ["RELIANCE", "TCS"])
    Returns a DataFrame of daily Close Price, columns = symbols, index = dates.
    """
    from_date = _to_ddmmyyyy(start)
    to_date = _to_ddmmyyyy(end)

    series = {}
    for sym in symbols:
        try:
            df = capital_market.price_volume_data(symbol=sym, from_date=from_date, to_date=to_date)
            if df is None or df.empty:
                logger.warning("No price data for %s", sym)
                continue

            df["Date"] = pd.to_datetime(df["Date"], format="mixed", dayfirst=True, errors="coerce")
            df = df.dropna(subset=["Date"])
            df = df.drop_duplicates(subset="Date", keep="last").set_index("Date").sort_index()

            close_col = "Close Price" if "Close Price" in df.columns else "ClosePrice"
            s = pd.to_numeric(
                df[close_col].astype(str).str.replace(",", "", regex=False), errors="coerce"
            )

            # Hard guarantee: collapse ANY remaining duplicate index labels,
            # regardless of why the earlier dedup missed them.
            if not s.index.is_unique:
                dupe_dates = s.index[s.index.duplicated()].unique().tolist()
                logger.warning(
                    "%s still had %d duplicate date(s) after drop_duplicates: %s — collapsing with groupby.last()",
                    sym, len(dupe_dates), dupe_dates,
                )
                s = s.groupby(level=0).last()
                print("Not unique")
                print(s.shape)
            series[sym] = s
        except Exception as exc:
            logger.warning("Price fetch failed for %s: %s", sym, exc)
        time.sleep(0.3)  # be polite to NSE

    if not series:
        return pd.DataFrame()

    # Final belt-and-suspenders check before DataFrame construction, with
    # a clear error message identifying the exact offending symbol(s)
    # instead of pandas' generic ValueError.
    bad_syms = [sym for sym, s in series.items() if not s.index.is_unique]
    if bad_syms:
        raise ValueError(f"Duplicate date index still present for symbols: {bad_syms}")
    
    prices = pd.DataFrame(series).ffill().dropna(how="all")
    logger.info("Fetched prices for %d/%d symbols.", len(series), len(symbols))
    print(prices.shape)
    return prices




def fetch_benchmark_prices(index_name: str, start: str, end: str) -> pd.Series:
    """
    Fetches historical OHLC for an NSE index (e.g. "NIFTY TOTAL MARKET",
    "NIFTY 500") via nselib.capital_market.index_data.

    NSE's underlying API caps each request to ~90 days of data, so this
    chunks the requested range into <=85-day windows and concatenates.
    """
    start_ts = pd.Timestamp(start)
    end_ts = pd.Timestamp(end)

    chunks = []
    cur_start = start_ts
    while cur_start <= end_ts:
        cur_end = min(cur_start + pd.Timedelta(days=85), end_ts)
        from_date = cur_start.strftime("%d-%m-%Y")
        to_date = cur_end.strftime("%d-%m-%Y")

        try:
            df = capital_market.index_data(index=index_name, from_date=from_date, to_date=to_date)
            if df is not None and not df.empty:
                chunks.append(df)
                print(f"Got {len(df)} rows for {index_name}: {from_date} -> {to_date}")
            else:
                print(f"Empty chunk for {index_name}: {from_date} -> {to_date}")
        except Exception as exc:
            print(f"Chunk fetch failed ({from_date} -> {to_date}): {exc}")

        time.sleep(0.5)  # avoid NSE rate limiting
        cur_start = cur_end + pd.Timedelta(days=1)

    if not chunks:
        raise RuntimeError(f"nselib returned no data for benchmark index {index_name!r}")

    df = pd.concat(chunks, ignore_index=True)
    print("Got benchmark data for", index_name)
    print(df.columns)

    df["Date"] = pd.to_datetime(df["TIMESTAMP"], format="mixed", dayfirst=True, errors="coerce")
    df = df.dropna(subset=["Date"])
    df = df.drop_duplicates(subset="Date", keep="last")  # dedupe overlapping chunk boundaries
    df = df.set_index("Date").sort_index()

    df = df.rename(columns={
        "OPEN_INDEX_VAL": "Open",
        "HIGH_INDEX_VAL": "High",
        "LOW_INDEX_VAL": "Low",
        "CLOSE_INDEX_VAL": "Close",
    })
    close_col = "Close" if "Close" in df.columns else "Close Price"
    print(df.shape)

    series = pd.to_numeric(df[close_col].astype(str).str.replace(",", "", regex=False), errors="coerce")
    series = series.groupby(series.index).last()  # extra safety against any remaining dupes
    return series


In [2]:
"""
metrics.py
----------
Computes Beta and annualised Standard Deviation (volatility) for each
stock in the universe, using trailing-twelve-month (TTM) daily returns.
"""

import logging
import numpy as np
import pandas as pd

from config import BENCHMARK_INDEX_NAME, TTM_TRADING_DAYS

logger = logging.getLogger("metrics")


def compute_risk_metrics(prices: pd.DataFrame, benchmark_prices: pd.Series) -> pd.DataFrame:
    """
    prices: DataFrame of daily close prices, columns = tickers, index = dates.
    benchmark_prices: Series of daily close prices for the benchmark index.

    Returns a DataFrame indexed by ticker with columns: Beta, StdDev_%
    """
    prices = prices.tail(TTM_TRADING_DAYS + 1)
    benchmark_prices = benchmark_prices.tail(TTM_TRADING_DAYS + 1)

    stock_returns = prices.pct_change().dropna(how="all")
    bench_returns = benchmark_prices.pct_change().dropna()

    aligned = stock_returns.join(bench_returns.rename("__bench__"), how="inner")
    bench_col = aligned["__bench__"]
    bench_var = bench_col.var()

    results = {}
    for ticker in prices.columns:
        col = aligned[ticker].dropna()
        common = aligned.loc[col.index, "__bench__"]
        if len(col) < 30 or bench_var == 0:
            beta = np.nan
        else:
            cov = np.cov(col, common)[0, 1]
            beta = cov / bench_var
        std_dev_annual_pct = col.std() * np.sqrt(252) * 100 if len(col) > 1 else np.nan
        results[ticker] = {"Beta": beta, "StdDev_%": std_dev_annual_pct}

    out = pd.DataFrame(results).T
    logger.info("Computed risk metrics for %d tickers.", len(out))
    return out


In [3]:
"""
screening.py
------------
Two-phase screening pipeline (matches the two-button Streamlit UI):

Phase 1 — run_fundamental_screen(): universe + Screener.in filters only.
Phase 2 — run_risk_screen(): takes the Phase 1 shortlist, fetches TTM
          prices via nselib, computes Beta & Std Dev, applies risk filters.
"""

import logging
import pandas as pd

from config import (
    build_screener_query,
    BETA_MAX,
    STD_DEV_MAX_PCT,
    BENCHMARK_INDEX_NAME,
    require_screener_session,
)
from universe import fetch_nifty_total_market, universe_symbols
from data_fetcher import fetch_screener_universe
from price_fetcher import fetch_stock_prices, fetch_benchmark_prices
from metrics import compute_risk_metrics

logger = logging.getLogger("screening")


def run_fundamental_screen(progress_cb=None) -> pd.DataFrame:
    def note(msg: str):
        logger.info(msg)
        if progress_cb:
            progress_cb(msg)

    require_screener_session()

    note("Fetching Nifty Total Market universe (750 stocks) …")
    universe_df = fetch_nifty_total_market()
    universe_set = set(universe_symbols(universe_df))
    note(f"Universe loaded: {len(universe_set)} symbols.")

    query = build_screener_query()
    note(f"Running Screener.in query: {query}")
    print(f"Running Screener.in query: {query}")
    fundamentals = fetch_screener_universe(query)
    print(fundamentals.columns)

    if fundamentals.empty:
        note("Screener.in returned zero rows — check SCREENER_SESSION and query syntax.")
        return pd.DataFrame()

    ticker_symbol = fundamentals["Ticker"].astype(str).str.replace(".NS", "", regex=False).str.upper()
    fundamentals = fundamentals[ticker_symbol.isin(universe_set)].copy()
    note(f"After restricting to Nifty Total Market universe: {len(fundamentals)} stocks pass fundamental filters.")
    return fundamentals


def run_risk_screen(fundamentals: pd.DataFrame, start_date: str, end_date: str, progress_cb=None) -> pd.DataFrame:
    def note(msg: str):
        logger.info(msg)
        if progress_cb:
            progress_cb(msg)

    if fundamentals.empty:
        return fundamentals

    symbols = (
        fundamentals["Ticker"].astype(str).str.replace(".NS", "", regex=False).str.upper().dropna().unique().tolist()
    )
    note(f"Downloading TTM daily prices for {len(symbols)} shortlisted stocks via nselib …")
    stock_prices = fetch_stock_prices(symbols, start=start_date, end=end_date)

    note(f"Downloading benchmark prices for {BENCHMARK_INDEX_NAME} …")
    benchmark_prices = fetch_benchmark_prices(BENCHMARK_INDEX_NAME, start=start_date, end=end_date)

    note("Computing Beta and annualised Std Dev (TTM) …")
    risk = compute_risk_metrics(stock_prices, benchmark_prices)
    risk.index = [f"{s}" for s in risk.index]

    fundamentals = fundamentals.copy()
    fundamentals["_sym"] = fundamentals["Ticker"].astype(str).str.replace(".NS", "", regex=False).str.upper()
    merged = fundamentals.merge(risk, left_on="_sym", right_index=True, how="left").drop(columns=["_sym"])
    return merged
    final = merged[(merged["Beta"] < BETA_MAX) & (merged["StdDev_%"] < STD_DEV_MAX_PCT)].copy()
    note(f"Final shortlist after Beta < {BETA_MAX} and StdDev < {STD_DEV_MAX_PCT}%: {len(final)} stocks.")

    cols = [c for c in [
        "Name", "Ticker", "Sector", "Market Cap", "P/E", "ROE", "ROCE",
        "Sales Growth", "Promoter Holding", "Beta", "StdDev_%"
    ] if c in final.columns]
    return final[cols].sort_values("Beta").reset_index(drop=True)


In [17]:
fundamentals_df = run_fundamental_screen()

Running Screener.in query: YOY Quarterly sales growth > 20 AND YOY Quarterly profit growth > 40 AND PEG Ratio < 1 AND Promoter holding > 60


Factor column(s) MISSING after normalisation: ['Sales Growth']
  Raw columns were: ['S.No.', 'Name', 'CMP Rs.', 'P/E', 'Market Cap', 'Sales Qtr Rs.Cr.', 'ROCE', 'Ind PE', 'PEG', 'EV Rs.Cr.', 'Piotroski score', 'Debt to Equity', 'ROE', 'ROCE 3Yr %', 'ROCE 5Yr %', 'Promoter Holding', 'Qtr Sales Var %', 'Qtr Profit Var %', 'Ticker']
  → Add the matching header as the first alias in EXPECTED_COLUMNS.
Universe has 356 > 200 stocks — sector enrichment skipped.


Index(['S.No.', 'Name', 'CMP Rs.', 'P/E', 'Market Cap', 'Sales Qtr Rs.Cr.',
       'ROCE', 'Ind PE', 'PEG', 'EV Rs.Cr.', 'Piotroski score',
       'Debt to Equity', 'ROE', 'ROCE 3Yr %', 'ROCE 5Yr %', 'Promoter Holding',
       'Qtr Sales Var %', 'Qtr Profit Var %', 'company_url', 'company_id',
       'Ticker', 'Sector', 'Sales Growth'],
      dtype='str')


In [18]:
fundamentals_df

,S.No.,Name,CMP Rs.,P/E,Market Cap,Sales Qtr Rs.Cr.,ROCE,Ind PE,PEG,EV Rs.Cr.,...,ROCE 3Yr %,ROCE 5Yr %,Promoter Holding,Qtr Sales Var %,Qtr Profit Var %,company_url,company_id,Ticker,Sector,Sales Growth
0,1.0,SPARC,235.20,4.88,7632.75,1853.22,165.00,33.58,0.05,8191.74,...,-80.64,-105.50,65.67,6715.81,2987.34,/company/SPARC/,SPARC,SPARC.NS,Unknown,NaN
229,230.0,Muthoot Finance,2989.50,11.33,120019.00,9288.71,15.77,21.78,0.26,258957.65,...,14.10,13.64,73.35,65.23,126.67,/company/MUTHOOTFIN/consolidated/,MUTHOOTFIN,MUTHOOTFIN.NS,Unknown,NaN
1,2.0,Tips Music,669.00,39.46,8551.93,103.93,122.19,43.65,0.95,8549.02,...,111.61,101.71,64.15,32.41,92.94,/company/TIPSMUSIC/,TIPSMUSIC,TIPSMUSIC.NS,Unknown,NaN
191,192.0,Viyash Scientific,268.65,58.76,11736.66,919.96,19.05,33.58,0.80,11956.71,...,9.32,6.17,61.31,129.02,455.37,/company/VIYASH/consolidated/,VIYASH,VIYASH.NS,Unknown,NaN
206,207.0,M R P L,174.49,10.99,30581.10,38254.19,17.69,5.76,-1.07,45311.53,...,15.94,16.38,88.58,120.41,317.07,/company/MRPL/consolidated/,MRPL,MRPL.NS,Unknown,NaN
332,333.0,Sheela Foam,761.10,55.20,8311.46,1050.06,6.11,33.07,-8.33,9183.05,...,5.87,9.80,65.69,23.59,511.54,/company/SFL/consolidated/,SFL,SFL.NS,Unknown,NaN
349,350.0,TARC Ltd,118.38,183.19,3493.35,208.70,2.26,27.86,-37.01,5307.20,...,-1.08,-0.50,65.12,1665.65,101.55,/company/TARC/consolidated/,TARC,TARC.NS,Unknown,NaN
343,344.0,Lloyds Enterpris,77.90,1077.47,11872.80,719.64,3.67,51.06,-27.08,11762.44,...,5.51,5.04,62.72,47.07,318.23,/company/LLOYDSENT/consolidated/,LLOYDSENT,LLOYDSENT.NS,Unknown,NaN
285,286.0,NLC India,295.60,11.64,40988.98,5042.46,10.45,23.40,0.74,68038.54,...,9.16,9.71,72.20,31.45,189.12,/company/NLCINDIA/consolidated/,NLCINDIA,NLCINDIA.NS,Unknown,NaN
289,290.0,R C F,127.83,17.82,7052.25,5580.57,10.23,15.51,-0.75,11027.59,...,7.96,12.05,75.00,49.63,123.99,/company/RCF/consolidated/,RCF,RCF.NS,Unknown,NaN


In [6]:
symbols = (
        fundamentals_df["Ticker"].astype(str).str.replace(".NS", "", regex=False).str.upper().dropna().unique().tolist()
    )

In [7]:
symbols

['SPARC',
 'MUTHOOTFIN',
 'VIYASH',
 'MRPL',
 'SFL',
 'TARC',
 'LLOYDSENT',
 'NLCINDIA',
 'RCF',
 'CENTURYPLY',
 'HUDCO',
 'MMTC',
 'FEDFINA',
 'TIPSMUSIC',
 'PREMIERENE',
 'UTLSOLAR',
 'QPOWER',
 'EMMVEE',
 'JSLL',
 'WAAREERTL',
 'HINDCOPPER',
 'WAAREEENER',
 'OSWALPUMPS',
 'KALYANKJIL',
 'PNGJL',
 'SENCO',
 'SKIPPER',
 'LLOYDSME',
 'THANGAMAYL',
 'SUPRIYA']

In [8]:
import datetime


In [9]:
end_date = datetime.date.today().isoformat()
start_date = (datetime.date.today() - datetime.timedelta(days=400)).isoformat()

In [10]:
symbols

['SPARC',
 'MUTHOOTFIN',
 'VIYASH',
 'MRPL',
 'SFL',
 'TARC',
 'LLOYDSENT',
 'NLCINDIA',
 'RCF',
 'CENTURYPLY',
 'HUDCO',
 'MMTC',
 'FEDFINA',
 'TIPSMUSIC',
 'PREMIERENE',
 'UTLSOLAR',
 'QPOWER',
 'EMMVEE',
 'JSLL',
 'WAAREERTL',
 'HINDCOPPER',
 'WAAREEENER',
 'OSWALPUMPS',
 'KALYANKJIL',
 'PNGJL',
 'SENCO',
 'SKIPPER',
 'LLOYDSME',
 'THANGAMAYL',
 'SUPRIYA']

In [14]:
stock_prices = fetch_stock_prices(symbols, start=start_date, end=end_date)

(271, 30)


In [15]:
stock_prices

,SPARC,MUTHOOTFIN,VIYASH,MRPL,SFL,TARC,LLOYDSENT,NLCINDIA,RCF,CENTURYPLY,...,HINDCOPPER,WAAREEENER,OSWALPUMPS,KALYANKJIL,PNGJL,SENCO,SKIPPER,LLOYDSME,THANGAMAYL,SUPRIYA
Date,,,,,,,,,,,,,,,,,,,,,
2025-06-13,165.90,2598.7,197.26,138.92,667.20,184.70,73.97,232.24,157.38,775.60,...,253.71,2824.7,NaN,518.95,593.70,347.30,510.65,1486.9,1852.2,689.55
2025-06-16,162.89,2633.3,198.25,136.67,706.95,187.37,72.77,231.01,158.44,759.70,...,262.76,2887.6,NaN,521.35,586.00,359.75,515.45,1518.4,1895.5,691.80
2025-06-17,159.88,2645.7,194.73,137.69,718.90,187.69,71.76,228.89,156.32,751.80,...,252.71,2790.4,NaN,514.55,581.00,352.15,500.10,1524.6,1883.6,680.60
2025-06-18,157.32,2634.4,193.96,135.65,745.85,187.37,70.72,226.88,156.29,750.10,...,253.93,2698.8,NaN,519.90,578.15,348.25,490.60,1503.1,1912.5,670.85
2025-06-19,150.62,2637.1,188.08,132.77,741.45,182.20,71.09,222.11,151.72,746.55,...,245.02,2671.2,NaN,511.05,568.85,339.05,475.60,1467.6,1882.5,655.25
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2026-07-13,259.42,3058.5,284.35,164.74,806.95,123.72,78.87,304.65,130.82,789.45,...,497.95,2880.2,408.00,510.65,598.35,381.00,556.00,1827.1,6233.0,845.00
2026-07-14,247.83,3016.4,278.90,156.96,808.95,121.62,77.82,303.60,128.47,781.65,...,491.65,2820.1,401.25,529.75,593.30,377.85,552.25,1825.5,6459.0,870.70
2026-07-15,251.04,3016.2,280.45,157.47,782.85,120.66,79.71,302.30,130.47,795.60,...,491.15,2812.8,405.80,548.10,595.45,373.20,545.15,1866.1,6617.0,852.40


In [16]:
run_risk_screen(fundamentals_df, start_date, end_date)

(271, 30)
Got 59 rows for NIFTY TOTAL MARKET: 13-06-2025 -> 06-09-2025
Got 58 rows for NIFTY TOTAL MARKET: 07-09-2025 -> 01-12-2025
Got 60 rows for NIFTY TOTAL MARKET: 02-12-2025 -> 25-02-2026
Got 56 rows for NIFTY TOTAL MARKET: 26-02-2026 -> 22-05-2026
Got 38 rows for NIFTY TOTAL MARKET: 23-05-2026 -> 18-07-2026
Got benchmark data for NIFTY TOTAL MARKET
Index(['INDEX_NAME', 'OPEN_INDEX_VAL', 'HIGH_INDEX_VAL', 'CLOSE_INDEX_VAL',
       'LOW_INDEX_VAL', 'TURN_OVER', 'TRADED_QTY', 'TIMESTAMP'],
      dtype='str')
(271, 8)


,S.No.,Name,CMP Rs.,P/E,Market Cap,Sales Qtr Rs.Cr.,ROCE,Ind PE,PEG,EV Rs.Cr.,...,Promoter Holding,Qtr Sales Var %,Qtr Profit Var %,company_url,company_id,Ticker,Sector,Sales Growth,Beta,StdDev_%
0,1.0,SPARC,235.20,4.88,7638.84,1853.22,165.00,33.47,0.05,8197.83,...,65.67,6715.81,2987.34,/company/SPARC/,SPARC,SPARC.NS,Unknown,NaN,1.837188,56.015050
230,231.0,Muthoot Finance,2989.50,11.33,120021.00,9288.71,15.77,21.78,0.26,258959.65,...,73.35,65.23,126.67,/company/MUTHOOTFIN/consolidated/,MUTHOOTFIN,MUTHOOTFIN.NS,Unknown,NaN,1.299086,36.602799
191,192.0,Viyash Scientific,268.65,58.82,11749.39,919.96,19.05,33.47,0.80,11969.44,...,61.31,129.02,455.37,/company/VIYASH/consolidated/,VIYASH,VIYASH.NS,Unknown,NaN,1.430156,42.421658
206,207.0,M R P L,174.49,10.96,30498.39,38254.19,17.69,5.76,-1.07,45228.82,...,88.58,120.41,317.07,/company/MRPL/consolidated/,MRPL,MRPL.NS,Unknown,NaN,1.199493,50.586658
333,334.0,Sheela Foam,761.10,55.17,8307.10,1050.06,6.11,32.99,-8.32,9178.69,...,65.69,23.59,511.54,/company/SFL/consolidated/,SFL,SFL.NS,Unknown,NaN,1.110477,37.931411
350,351.0,TARC Ltd,118.38,183.53,3499.86,208.70,2.26,27.87,-37.08,5313.71,...,65.12,1665.65,101.55,/company/TARC/consolidated/,TARC,TARC.NS,Unknown,NaN,1.460976,41.586932
344,345.0,Lloyds Enterpris,77.90,1076.24,11859.32,719.64,3.67,50.95,-27.05,11748.96,...,62.72,47.07,318.23,/company/LLOYDSENT/consolidated/,LLOYDSENT,LLOYDSENT.NS,Unknown,NaN,2.097638,53.978651
286,287.0,NLC India,295.60,11.65,41050.41,5042.46,10.45,23.40,0.74,68099.97,...,72.20,31.45,189.12,/company/NLCINDIA/consolidated/,NLCINDIA,NLCINDIA.NS,Unknown,NaN,0.996608,37.451533
290,291.0,R C F,127.83,17.83,7056.64,5580.57,10.23,15.50,-0.75,11031.98,...,75.00,49.63,123.99,/company/RCF/consolidated/,RCF,RCF.NS,Unknown,NaN,1.537425,34.734129
275,276.0,Century Plyboard,796.05,65.87,17671.52,1492.21,11.45,40.42,-5.43,19377.39,...,71.83,24.52,48.92,/company/CENTURYPLY/consolidated/,CENTURYPLY,CENTURYPLY.NS,Unknown,NaN,0.923354,27.644647


In [55]:
benchmark_prices = fetch_benchmark_prices("NIFTY 500", start=start_date, end=end_date)

Got 60 rows for NIFTY 500: 12-06-2025 -> 05-09-2025
Got 57 rows for NIFTY 500: 06-09-2025 -> 30-11-2025
Got 60 rows for NIFTY 500: 01-12-2025 -> 24-02-2026
Got 56 rows for NIFTY 500: 25-02-2026 -> 21-05-2026
Got 39 rows for NIFTY 500: 22-05-2026 -> 17-07-2026
Got benchmark data for NIFTY 500
Index(['INDEX_NAME', 'OPEN_INDEX_VAL', 'HIGH_INDEX_VAL', 'CLOSE_INDEX_VAL',
       'LOW_INDEX_VAL', 'TURN_OVER', 'TRADED_QTY', 'TIMESTAMP'],
      dtype='str')
(272, 8)


In [22]:
benchmark_prices

Date
2025-06-11    13141.95
2025-06-12    12976.60
2025-06-13    12898.95
2025-06-16    13002.70
2025-06-17    12936.25
                ...   
2026-07-10    13177.00
2026-07-13    13177.05
2026-07-14    13091.65
2026-07-15    13130.05
2026-07-16    13110.50
Name: Close, Length: 95, dtype: float64

In [40]:
benchmark_prices

Date
2025-06-11    13141.95
2025-06-12    12976.60
2025-06-13    12898.95
2025-06-16    13002.70
2025-06-17    12936.25
                ...   
2026-07-10    13177.00
2026-07-13    13177.05
2026-07-14    13091.65
2026-07-15    13130.05
2026-07-16    13110.50
Name: Close, Length: 95, dtype: float64

In [43]:
pd.DataFrame(benchmark_prices).reset_index()

,Date,Close
0,2025-06-11,13141.95
1,2025-06-12,12976.60
2,2025-06-13,12898.95
3,2025-06-16,13002.70
4,2025-06-17,12936.25
...,...,...
90,2026-07-10,13177.00
91,2026-07-13,13177.05
92,2026-07-14,13091.65
93,2026-07-15,13130.05


In [44]:
start_date

'2025-06-12'

In [45]:
end_date

'2026-07-17'

In [47]:
date_range = pd.date_range(start_date, end_date, freq='B')

In [48]:
date_range

DatetimeIndex(['2025-06-12', '2025-06-13', '2025-06-16', '2025-06-17',
               '2025-06-18', '2025-06-19', '2025-06-20', '2025-06-23',
               '2025-06-24', '2025-06-25',
               ...
               '2026-07-06', '2026-07-07', '2026-07-08', '2026-07-09',
               '2026-07-10', '2026-07-13', '2026-07-14', '2026-07-15',
               '2026-07-16', '2026-07-17'],
              dtype='datetime64[us]', length=287, freq='B')

In [50]:
df = pd.DataFrame(benchmark_prices.reindex(date_range))

In [52]:
df[df['Close'].isna()].index

DatetimeIndex(['2025-08-15', '2025-08-27', '2025-09-19', '2025-09-22',
               '2025-09-23', '2025-09-24', '2025-09-25', '2025-09-26',
               '2025-09-29', '2025-09-30',
               ...
               '2026-06-01', '2026-06-02', '2026-06-03', '2026-06-04',
               '2026-06-05', '2026-06-08', '2026-06-09', '2026-06-10',
               '2026-06-26', '2026-07-17'],
              dtype='datetime64[us]', length=193, freq=None)